# Notebook 6 — Model Serialization

## Objetivo

Convertir el modelo evaluado en el Notebook 5 en un **artefacto reproducible y preparado para producción**.

En esta etapa no entrenamos ni optimizamos el modelo.

El objetivo es dejar preparado un paquete que pueda ser consumido posteriormente por:

- un endpoint online;
- una inferencia batch;
- una aplicación;
- Power BI;
- un proceso de monitoreo.

### Flujo profesional

```text
05 Model Evaluation
        ↓
06 Model Serialization
        ↓
   Modelo + Metadata
        ↓
07 Deployment
        ↓
08 Batch Inference
        ↓
09 Monitoring
        ↓
Power BI / Apps
```

### Principio

No basta con guardar un `.pkl`.

Para producción necesitamos conocer:

- qué modelo se guardó;
- qué variables espera;
- qué versión de Python/scikit-learn se utilizó;
- qué artefacto fue evaluado;
- cuándo se generó;
- cómo verificar que el archivo no fue alterado.

## 01. Librerías

Utilizamos únicamente librerías compatibles con el entorno actual de Azure Machine Learning.

No instalamos paquetes nuevos ni modificamos el Compute.

`joblib` permite serializar el Pipeline completo y `json` permite guardar la metadata necesaria para reproducibilidad.

In [1]:
# ==========================================
# 01. Librerías
# ==========================================

import os
import json
import hashlib
import platform
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn

print("Librerías cargadas correctamente.")
print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

Librerías cargadas correctamente.
Python: 3.10.0
numpy: 1.23.5
pandas: 1.5.3
scikit-learn: 1.5.1
joblib: 1.2.0


## 02. Configuración de rutas

El Notebook 4 generó el candidato optimizado:

```text
../models/tuned_random_forest.pkl
```

El Notebook 5 lo evaluó sobre el Test Set y generó la evidencia de desempeño.

Aquí creamos un artefacto de producción separado:

```text
../models/production_model.pkl
```

De esta manera conservamos el artefacto original y dejamos explícito cuál es el modelo destinado a las siguientes etapas.

In [2]:
# ==========================================
# 02. Configuración
# ==========================================

MODEL_SOURCE_PATH = "../models/tuned_random_forest.pkl"
PRODUCTION_MODEL_PATH = "../models/production_model.pkl"

EVALUATION_METRICS_PATH = "../models/model_evaluation_metrics.csv"
CLASSIFICATION_REPORT_PATH = "../models/model_classification_report.csv"
PREDICTIONS_PATH = "../models/model_test_predictions.csv"

FEATURE_DATA_PATH = "../data/output_data/featureengineering_files.parquet"

METADATA_PATH = "../models/model_metadata.json"
FEATURE_SCHEMA_PATH = "../models/model_feature_schema.json"
MANIFEST_PATH = "../models/model_manifest.json"

RANDOM_STATE = 42
TARGET = "label_e"

REMOVE_COLUMNS = [
    "machineID",
    "dt_truncated",
    "datetime",
    "failure",
    "model",
    TARGET
]

os.makedirs("../models", exist_ok=True)

print("Modelo fuente:", MODEL_SOURCE_PATH)
print("Modelo producción:", PRODUCTION_MODEL_PATH)

## 03. Verificación de artefactos previos

Antes de serializar verificamos que existan:

1. El modelo optimizado.
2. Las métricas de evaluación.
3. El reporte por clase.
4. Las predicciones de Test.

Esto introduce una práctica importante de MLOps:

> **No promover un modelo a producción sin evidencia de evaluación.**

In [3]:
# ==========================================
# 03. Verificación
# ==========================================

required_files = {
    "Modelo optimizado": MODEL_SOURCE_PATH,
    "Métricas evaluación": EVALUATION_METRICS_PATH,
    "Classification report": CLASSIFICATION_REPORT_PATH,
    "Predicciones Test": PREDICTIONS_PATH
}

missing = [
    name
    for name, path in required_files.items()
    if not os.path.exists(path)
]

if missing:
    raise FileNotFoundError(
        "Faltan artefactos requeridos: " + ", ".join(missing)
    )

print("Todos los artefactos requeridos están disponibles.")

## 04. Carga y validación del modelo

Cargamos el modelo optimizado y comprobamos que sea un Pipeline de inferencia válido.

La ventaja de haber utilizado un Pipeline es que el artefacto conserva conjuntamente el proceso de transformación y el estimador.

Esto reduce el riesgo de que producción aplique una transformación diferente a la utilizada durante entrenamiento.

In [ ]:
# ==========================================
# 04. Carga y validación
# ==========================================

model = joblib.load(MODEL_SOURCE_PATH)

print("Tipo de objeto:")
print(type(model))

print()
print("Pasos del Pipeline:")

if hasattr(model, "named_steps"):
    for step_name, step in model.named_steps.items():
        print(f"- {step_name}: {type(step).__name__}")
else:
    print("ADVERTENCIA: el objeto no expone named_steps.")

print()
print("El modelo fue cargado correctamente.")

## 05. Recuperación del esquema de entrada

Un endpoint necesita saber exactamente qué variables debe recibir.

Construimos el esquema a partir del dataset utilizado en el proyecto.

Guardaremos:

- nombre de cada variable;
- tipo de dato;
- cantidad de variables;
- orden esperado.

### Importante

El orden y los nombres de las variables forman parte del contrato de inferencia.

Si posteriormente cambia el esquema de entrada, el modelo debe versionarse y evaluarse nuevamente.

In [ ]:
# ==========================================
# 05. Esquema de entrada
# ==========================================

if not os.path.exists(FEATURE_DATA_PATH):
    raise FileNotFoundError(
        f"No se encontró el dataset: {FEATURE_DATA_PATH}"
    )

data = pd.read_parquet(FEATURE_DATA_PATH)

feature_columns = [
    col
    for col in data.columns
    if col not in REMOVE_COLUMNS
]

feature_schema = []

for position, column in enumerate(feature_columns):
    feature_schema.append({
        "position": position,
        "name": column,
        "dtype": str(data[column].dtype)
    })

print("Número de variables:", len(feature_columns))
print()
print("Primeras variables:")

display(
    pd.DataFrame(feature_schema).head(20)
)

## 06. Validación de inferencia

Antes de promover el artefacto hacemos una prueba real de inferencia.

Tomamos algunas observaciones del dataset y verificamos:

- que `predict()` funcione;
- que `predict_proba()` funcione;
- que las probabilidades sean válidas;
- que el número de clases sea consistente.

Esta prueba funciona como un **smoke test** del artefacto.

In [ ]:
# ==========================================
# 06. Smoke test
# ==========================================

X_sample = data[feature_columns].head(5).copy()

sample_predictions = model.predict(X_sample)
sample_probabilities = model.predict_proba(X_sample)

if len(sample_predictions) != len(X_sample):
    raise RuntimeError(
        "El número de predicciones no coincide con las entradas."
    )

if sample_probabilities.shape[0] != len(X_sample):
    raise RuntimeError(
        "El número de filas de predict_proba no coincide."
    )

if not np.allclose(
    sample_probabilities.sum(axis=1),
    1.0,
    atol=1e-6
):
    raise RuntimeError(
        "Las probabilidades no suman 1."
    )

print("Smoke test exitoso.")
print("Predicciones:", sample_predictions.tolist())
print("Shape probabilidades:", sample_probabilities.shape)

## 07. Lectura de métricas finales

La serialización debe estar asociada a la evidencia que justificó la promoción del modelo.

Leemos las métricas generadas en el Notebook 5 para incorporarlas a la metadata.

Esto permite responder posteriormente:

> "¿Con qué desempeño fue promovido este modelo?"

In [ ]:
# ==========================================
# 07. Métricas de evaluación
# ==========================================

evaluation_metrics = pd.read_csv(
    EVALUATION_METRICS_PATH
)

display(evaluation_metrics.round(4))

## 08. Promoción del modelo

Copiamos el modelo evaluado a un nombre explícito de producción:

```text
production_model.pkl
```

No modificamos el contenido del Pipeline.

El archivo es simplemente una copia del artefacto seleccionado y evaluado.

Esto permite conservar:

```text
tuned_random_forest.pkl
        ↓
production_model.pkl
```

y mantener trazabilidad entre el modelo optimizado y el modelo promovido.

In [8]:
# ==========================================
# 08. Promoción del modelo
# ==========================================

joblib.dump(
    model,
    PRODUCTION_MODEL_PATH
)

if not os.path.exists(PRODUCTION_MODEL_PATH):
    raise RuntimeError(
        "No fue posible crear el modelo de producción."
    )

source_size = os.path.getsize(MODEL_SOURCE_PATH)
production_size = os.path.getsize(PRODUCTION_MODEL_PATH)

print("Modelo de producción creado.")
print("Archivo:", PRODUCTION_MODEL_PATH)
print(f"Tamaño fuente      : {source_size:,} bytes")
print(f"Tamaño producción  : {production_size:,} bytes")

Modelo de producción creado.
Archivo: ../models/production_model.pkl
Tamaño fuente      : 82,344,371 bytes
Tamaño producción  : 82,344,627 bytes


## 09. Integridad del artefacto

Calculamos un hash SHA-256 del archivo.

El hash permite verificar posteriormente que el artefacto utilizado en deployment es exactamente el mismo archivo que fue validado durante esta etapa.

```text
Modelo
  ↓
SHA-256
  ↓
Manifest
```

Es una práctica sencilla pero útil para trazabilidad.

In [9]:
# ==========================================
# 09. Hash del modelo
# ==========================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

model_hash = sha256_file(PRODUCTION_MODEL_PATH)

print("SHA-256:")
print(model_hash)

SHA-256:
bff6b4a0be55a7db649716ee1ac7b8981f360e463ed63451f5a32d4173fb67bd


## 10. Metadata del modelo

Creamos un archivo de metadata que acompaña al `.pkl`.

La metadata documenta:

- nombre del modelo;
- algoritmo;
- variable objetivo;
- número de features;
- versión de Python;
- versión de scikit-learn;
- versión de numpy;
- versión de pandas;
- fecha de serialización;
- métrica principal;
- hash del artefacto;
- ruta del artefacto original.

Esto facilita auditoría, troubleshooting y promoción entre ambientes.

In [10]:
# ==========================================
# 10. Metadata
# ==========================================

metric_row = evaluation_metrics.iloc[0].to_dict()

metadata = {
    "model_name": "Random Forest",
    "model_artifact": "production_model.pkl",
    "source_artifact": os.path.basename(MODEL_SOURCE_PATH),
    "task": "multiclass_classification",
    "target": TARGET,
    "primary_metric": "F1 Macro",
    "primary_metric_value": float(
        metric_row.get("F1 Macro", np.nan)
    ),
    "accuracy": float(
        metric_row.get("Accuracy", np.nan)
    ),
    "precision_macro": float(
        metric_row.get("Precision Macro", np.nan)
    ),
    "recall_macro": float(
        metric_row.get("Recall Macro", np.nan)
    ),
    "roc_auc_macro_ovr": float(
        metric_row.get("ROC-AUC Macro OVR", np.nan)
    ),
    "n_features": len(feature_columns),
    "n_classes": int(data[TARGET].nunique()),
    "classes": sorted(
        [int(x) if isinstance(x, (np.integer, int)) else str(x)
         for x in data[TARGET].unique()]
    ),
    "random_state": RANDOM_STATE,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
    "serialized_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "sha256": model_hash
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Metadata guardada:")
print(METADATA_PATH)

Metadata guardada:
../models/model_metadata.json


## 11. Guardar esquema de features

Separamos el contrato de entrada de la metadata general.

El archivo permitirá que el futuro endpoint o proceso batch conozca qué variables espera el modelo.

```text
model_feature_schema.json
```

Este archivo será especialmente útil cuando construyamos el endpoint del Notebook 7.

In [11]:
# ==========================================
# 11. Esquema persistente
# ==========================================

schema_document = {
    "model": "production_model.pkl",
    "target": TARGET,
    "n_features": len(feature_columns),
    "features": feature_schema
}

with open(
    FEATURE_SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        schema_document,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Esquema guardado:")
print(FEATURE_SCHEMA_PATH)

## 12. Manifest de producción

El manifest funciona como un índice del paquete.

Permite identificar rápidamente los artefactos asociados al modelo:

```text
production_model.pkl
model_metadata.json
model_feature_schema.json
model_manifest.json
```

En proyectos reales este concepto puede evolucionar hacia un Model Registry, pero para este laboratorio nos permite demostrar correctamente el principio de versionado y trazabilidad.

In [ ]:
# ==========================================
# 12. Manifest
# ==========================================

manifest = {
    "model_artifact": os.path.basename(PRODUCTION_MODEL_PATH),
    "metadata_artifact": os.path.basename(METADATA_PATH),
    "feature_schema_artifact": os.path.basename(
        FEATURE_SCHEMA_PATH
    ),
    "evaluation_artifact": os.path.basename(
        EVALUATION_METRICS_PATH
    ),
    "model_sha256": model_hash,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "validated_for_next_stage"
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Manifest guardado:")
print(MANIFEST_PATH)

## 13. Validación final del paquete

Volvemos a cargar el modelo desde el archivo de producción.

Esto es importante.

No queremos demostrar únicamente que el objeto existente funciona; queremos comprobar que el **artefacto serializado y posteriormente cargado** funciona.

Flujo:

```text
joblib.dump()
     ↓
production_model.pkl
     ↓
joblib.load()
     ↓
predict()
     ↓
OK
```

In [ ]:
# ==========================================
# 13. Validación final
# ==========================================

reloaded_model = joblib.load(
    PRODUCTION_MODEL_PATH
)

reloaded_predictions = reloaded_model.predict(
    X_sample
)

if not np.array_equal(
    sample_predictions,
    reloaded_predictions
):
    raise RuntimeError(
        "El modelo recargado no reproduce "
        "las mismas predicciones."
    )

reloaded_hash = sha256_file(
    PRODUCTION_MODEL_PATH
)

if reloaded_hash != model_hash:
    raise RuntimeError(
        "El hash del modelo cambió inesperadamente."
    )

print("VALIDACIÓN FINAL EXITOSA")
print("✓ Modelo serializado")
print("✓ Modelo recargado")
print("✓ Predicciones reproducibles")
print("✓ Hash verificado")

## 14. Inventario de artefactos

Al finalizar esta etapa tendremos:

```text
models/
│
├── tuned_random_forest.pkl
│
├── production_model.pkl
├── model_metadata.json
├── model_feature_schema.json
├── model_manifest.json
│
├── model_evaluation_metrics.csv
├── model_classification_report.csv
└── model_test_predictions.csv
```

### ¿Qué pasa ahora?

El modelo deja de ser solamente un resultado experimental y pasa a ser un **artefacto consumible por infraestructura de inferencia**.

La siguiente etapa será:

```text
06 Model Serialization
        ↓
07 Deployment
        ↓
Endpoint Online
        ↓
Predicción desde una aplicación
```

In [ ]:
# ==========================================
# 14. Inventario
# ==========================================

artifacts = [
    PRODUCTION_MODEL_PATH,
    METADATA_PATH,
    FEATURE_SCHEMA_PATH,
    MANIFEST_PATH,
    EVALUATION_METRICS_PATH,
    CLASSIFICATION_REPORT_PATH,
    PREDICTIONS_PATH
]

inventory = []

for path in artifacts:
    inventory.append({
        "artifact": os.path.basename(path),
        "path": path,
        "exists": os.path.exists(path),
        "size_bytes": (
            os.path.getsize(path)
            if os.path.exists(path)
            else None
        )
    })

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

# Resumen del Notebook 6

Al finalizar esta etapa tendremos:

```text
✓ Modelo optimizado identificado
✓ Modelo de producción serializado
✓ Smoke test de inferencia
✓ Esquema de entrada documentado
✓ Metadata técnica
✓ Métricas asociadas al modelo
✓ Hash SHA-256
✓ Manifest de artefactos
✓ Validación después de reload
✓ Paquete preparado para deployment
```

## Siguiente etapa

```text
06 Model Serialization
          ↓
07 Deployment
          ↓
08 Batch Inference
          ↓
09 Monitoring
          ↓
Power BI / Apps
```

### Concepto clave

**Serializar no es simplemente guardar un `.pkl`.**

Es convertir un modelo experimental en un artefacto:

**reproducible + verificable + documentado + consumible.**